# Hourly Series Analysis

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import mlflow
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from eda_utils import line_with_slider, plot_decomposition, plot_with_residuals
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing as HoltWinters
from statsmodels.tsa.seasonal import MSTL, STL, seasonal_decompose

from solar_predictor.data import drop_incomplete_tail, read_sarah3
from solar_predictor.metrics import calculate_metrics
from solar_predictor.storage import get_location_prefix
from solar_predictor.utils import LOCATIONS

px.defaults.height = 700

/home/mario/workspace/solar-predictor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# skopje_lp = get_location_prefix(LOCATIONS["SKOPJE"]["LAT"], LOCATIONS["SKOPJE"]["LONG"])
# raw_df = read_sarah3(skopje_lp)
# raw_df.to_csv("raw_df.csv")

raw_df = pd.read_csv("raw_df.csv", parse_dates=["datetime"]).set_index("datetime")
df = drop_incomplete_tail(raw_df)

# Trend, Seasonal and Residual Decomposition

Hourly radiation has two cycles: the day (24 hours) and the year (24 * 365 hours), so MSTL is used to separate both.
Only the last 5 years are used to keep the computation fast.

In [ ]:
hourly_df = df[["shortwave_radiation"]].resample("h").mean()
hourly_df = hourly_df.loc[hourly_df.index.max() - pd.DateOffset(years=10) :]
hourly_df["shortwave_radiation"] = hourly_df["shortwave_radiation"].interpolate(
    method="time"
)

res = MSTL(hourly_df["shortwave_radiation"], periods=(24, 24 * 365), iterate=1).fit()

In [7]:
last_n = 5000
fig = plot_decomposition(
    res.trend.iloc[-last_n:],
    res.seasonal.sum(axis=1).iloc[-last_n:],
    res.resid.iloc[-last_n:],
    window=pd.DateOffset(weeks=2),
    title="MSTL decomposition (periods=24, 8760)",
)
fig.show()

fig = line_with_slider(df=res.seasonal.iloc[-last_n:], window=pd.DateOffset(weeks=2))
fig.update_layout(title="Daily and yearly seasonal components")
fig.show()